In [1]:
from pyscripts.DEV_main import *
import flax.linen as nn

jax:    0.4.35
jaxlib: 0.4.34
numpy:  1.26.4
python: 3.12.7 | packaged by conda-forge | (main, Oct  4 2024, 16:05:46) [GCC 13.3.0]
device info: NVIDIA A100-SXM4-40GB-1, 1 local devices"
process_count: 1
platform: uname_result(system='Linux', node='evidently-classic-hyena', release='6.8.0-48-generic', version='#48-Ubuntu SMP PREEMPT_DYNAMIC Fri Sep 27 14:04:52 UTC 2024', machine='x86_64')


$ nvidia-smi
Tue Nov 12 21:18:06 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.06             Driver Version: 535.183.06   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=====================

In [2]:
json_params = {"fname_dcd": "/media/volume/Joseph-Large/githubs/Deep-MMS/Simulation/1crn_split2.dcd",
               "fname_prmtop": "/media/volume/Joseph-Large/githubs/Deep-MMS/Simulation/1crn_H.prmtop",
               "fname_pdb": "None",
               "save_dir": "/media/volume/Joseph-Large2/githubs/Deep-MMS/",
               "data_dir": "None",
               "max_epoch": 300000,
               "latent_dim": 64,
               "test_slice": 3,
               "data_slice_start": 0,
               "data_slice_end": "None",
               "model_name": "1crn_AE_Dropout_RMSD_and_Scale_Components",
               "batch_size": 1000,
               "learning_rate": 1e-4,
               "dropout_rates": [0.5, 0.4, 0.4, 0.3, 0.3, 0.2, 0.2, 0.1, 0.1, 0.1],
               "potential_threshold": 0.05,
               "model_type": "AE",
               "coordinate_scheme": "Cartesian",
               "resume_latest": False,
               "report_potential": True,
               "checkpoint_interval": 200,
               "scale_factor": 1.0}

In [3]:
self = NN_Experiment(json_params, from_json_params=True)
file = open("output.out", 'w')

Load Files
Establish Directories
Load Data with MDTraj
Coordinates
Batch data
(8000, 1926) (2000, 1926)
Model Init


Checkpointer
Batch Data
Establish NCs
Epoch 0
NC <class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    dimensions(sizes): 
    variables(dimensions): 
    groups: Train, Test
Model BatchNorm_VAE(
    # attributes
    input_size = 1926
    hidden_layers = [1926, 1798, 1656, 1497, 1319, 1120, 898, 650, 373, 64]
    latents = 64
    dropout_rates = [0.5, 0.4, 0.4, 0.3, 0.3, 0.2, 0.2, 0.1, 0.1, 0.1]
)
INITIALIZATION COMPLETE for 64 Latents, 1.0 Scale


In [4]:
from pyscripts.DEV_loss import *

In [5]:
loss_evals = [atom_rmsd, sum1_loss, sum2_loss, sum3_loss, sum4_loss, sum5_loss, sum6_loss, sum7_loss]

In [ ]:
for i in range(len(loss_evals)):
    print(f"Loss function {i}", file=file, flush=True)
    loss_eval = loss_evals[i]
    loss = 1e5
    while loss > 1:
        epoch_start = datetime.now()
        
        #Training
        f = iter(self.train_batches)
        batch_loss = []
        for b in range(self.train_batches.num_batches):
            #Get Batch
            batch = next(f)
            #Train Batch
            root_key = jax.random.PRNGKey(self.epoch)
            main_key, params_key, dropout_key = jax.random.split(key=root_key, num=3)
            self.state, loss = step(self.state, batch, root_key, dropout_key, loss_eval)
            batch_loss.append(loss)
        train_loss = np.mean(batch_loss)
        
        #After ANY EPOCH
        save_args = orbax_utils.save_args_from_target(self.state)
        self.checkpoint_manager.save(self.epoch, self.state, save_kwargs={'save_args': save_args})
            
        #After all batches seen this epoch
        epoch_end = datetime.now() - epoch_start
        if self.epoch % 10 == 0:
            recon = self.state.apply_fn({'params':self.state.params, 'batch_stats':self.state.batch_stats}, self.test_data, root_key, train=False, rngs={'dropout': dropout_key})[0]
            test_loss = loss_eval(self.test_data, recon).mean()
            print(f"{self.epoch} - {'%.4E'%train_loss} - {'%.4E'%test_loss} - 'Time:'{epoch_end}", file=file, flush=True)
        
        self.epoch += 1
    #Print the data after this loss converged
    print("#####  END THIS LOSS FUNCTION WITH LOSS #####", file=file, flush=True)
    recon = self.state.apply_fn({'params':self.state.params, 'batch_stats':self.state.batch_stats}, self.test_data, root_key, train=False, rngs={'dropout': dropout_key})[0]
    test_loss = loss_eval(self.test_data, recon).mean()
    print(f"{self.epoch} - {'%.4E'%train_loss} - {'%.4E'%test_loss} - 'Time:'{epoch_end}", file=file, flush=True)
    print("#############################################", file=file, flush=True)

    #Write a structure (fuckitmaybeitsinteresting)
    fname = f"loss_function{i}_ReconTest.dcd"
    with md.formats.DCDTrajectoryFile(fname, 'w') as f:
        traj_xyz = recon.reshape(recon.shape[0], -1, 3)
        f.write(traj_xyz*10)

In [ ]:
raise Exception('You Shall Not Pass!')

In [ ]:
experiment.MAIN_scale_and_train_potential(n_rmsd=200)

In [ ]:
experiment.train_scaling_potential()

In [ ]:
raise Exception('So you have chosen death...')

In [ ]:
traingrp = experiment.rootgrp['Train']
testgrp = experiment.rootgrp['Test']
scaling_start = np.where(experiment.rootgrp['Train'].variables['Potential Coefficient'][:] == 0)[0][-1]
scaling_end = np.where(experiment.rootgrp['Train'].variables['Potential Coefficient'][:] == 1)[0][0]

In [ ]:
plt.clf()
_ = plt.plot(np.arange(experiment.epoch), np.mean(traingrp['RMSD'][:-1, :], axis=1))
_ = plt.plot(np.arange(experiment.epoch), np.mean(testgrp['RMSD'][:-1, :], axis=1))
plt.legend(['Train', 'Test'])
plt.show()

In [ ]:
key='Potential'
plt.clf()
_ = plt.plot(np.arange(experiment.epoch), np.mean(traingrp[key][:-1, :], axis=1))
_ = plt.plot(np.arange(experiment.epoch), np.mean(testgrp[key][:-1, :], axis=1))
plt.legend(['Train', 'Test'])
plt.yscale('log')
plt.show()

In [ ]:
key='Potential'
plt.clf()
_ = plt.plot(np.arange(10000, experiment.epoch), np.mean(traingrp[key][10000:-1, :], axis=1))
_ = plt.plot(np.arange(10000, experiment.epoch), np.mean(testgrp[key][10000:-1, :], axis=1))
plt.legend(['Train', 'Test'])
plt.yscale('log')
plt.hlines(10, 10000, 30000, colors='red', linestyles='dashed')
plt.show()

In [ ]:
def epoch_would_have_stopped(potential_threshold):
    key = 'Potential'
    num_move_ave = 100
    train_potentials = np.mean(traingrp[key][:-1, :], axis=1)
    test_potentials = np.mean(testgrp[key][:-1, :], axis=1)
    i = 100
    potential_above_threshold, test_potential_decreasing = True, True
    while potential_above_threshold or test_potential_decreasing:
        potential_above_threshold = (train_potentials[i-num_move_ave:i].mean() > potential_threshold or
                                     test_potentials[i-num_move_ave:i].mean() > potential_threshold)
        test_potential_decreasing = test_potentials[i-2*num_move_ave:i-num_move_ave].mean() > test_potentials[i-num_move_ave:i].mean()
        i += 1
    return i

In [ ]:
experiment.epoch

In [ ]:
epoch_would_have_stopped(10)

In [ ]:
key='Potential'
plt.clf()
_ = plt.plot(np.arange(10000, experiment.epoch), np.mean(traingrp[key][10000:-1, :], axis=1))
_ = plt.plot(np.arange(10000, experiment.epoch), np.mean(testgrp[key][10000:-1, :], axis=1))
plt.legend(['Train', 'Test'])
plt.yscale('log')
plt.hlines(10, 10000, 30000, colors='red', linestyles='dashed')
plt.vlines(epoch_would_have_stopped(10), 1, 1000, colors='red', linestyles='dashed')
plt.show()

In [ ]:
#Attempt to resume a Bridges2 run with potential scaling

In [ ]:
rng = jax.random.PRNGKey(experiment.epoch)
rng, key = jax.random.split(rng)
recon, latents = experiment.state.apply_fn({'params':experiment.state.params}, experiment.test_data, rng)

fig, axs = plt.subplots(experiment.n_latents, experiment.n_latents, figsize=(15,10), sharex='col')
for i in range(experiment.n_latents):
    for j in range(experiment.n_latents):
        #axs[i,j].scatter(latent[:,j], latent[:,i])
        if i > j:
            axs[i,j].scatter(latents[:,j], latents[:,i])
        elif i == j:
            axs[i,j].hist(latents[:, i], bins=25)

In [ ]:
print(experiment.rootgrp)
print(experiment.rootgrp['Train'])
print(experiment.rootgrp['Test'])

In [ ]:
plt.clf()
_ = plt.plot(np.arange(rmsd_train.shape[0]), rmsd_train)
_ = plt.plot(np.arange(rmsd_test.shape[0]), rmsd_test)
plt.legend(['Train', 'Test'])
plt.ylim(0.14, .15)
plt.show()

In [ ]:
rmsd_train = np.mean(experiment.rootgrp['Train'].variables['RMSD'], axis=1)
rmsd_test = np.mean(experiment.rootgrp['Test'].variables['RMSD'], axis=1)
d_tt = np.sqrt((rmsd_test - rmsd_train)**2)/rmsd_train * 100

In [ ]:
plt.clf()
_ = plt.plot(np.arange(d_tt.shape[0]), d_tt)
plt.xlabel('Epoch')
plt.ylabel('Deviation (Percentage)')
plt.title('Percent Deviation of Test Loss from Train Loss')
plt.show()

In [ ]:
rng_init = jax.random.PRNGKey(54)
rng, key = jax.random.split(rng_init)
test_data = experiment.test_data
recon, latents = experiment.state.apply_fn({'params':experiment.state.params}, test_data, rng)

In [ ]:
for i in range(latents.shape[-1]):
    plt.clf()
    plt.title(f'Latent {i}')
    _ = plt.hist(latents[:, i], bins=100)
    plt.show()

In [ ]:
from sklearn.mixture import GaussianMixture
ics = []
for i in range(1, 50):
    X = np.array(latents)
    MM = GaussianMixture(n_components=i).fit(X)
    ics.append((i, MM.aic(X), MM.bic(X)))
ics = np.array(ics)
plt.clf()
_ = plt.plot(ics[:, 0], ics[:, 1])
_ = plt.plot(ics[:, 0], ics[:, 2])
plt.legend(('Akaike Info Criterion', 'Bayes Info Criterion'))
plt.xlabel('Num Components')
plt.show()
print(f'minimums: Akaike {np.argmin(ics[:,1]) + 1} Bayes {np.argmin(ics[:,2]) + 1}')

In [ ]:
X = np.array(latents)
MM = GaussianMixture(n_components=2).fit(X) #chosen based on above graph
samples = MM.sample(2000)[0]

In [ ]:
for i in range(samples.shape[-1]):
    plt.clf()
    _ = plt.hist(latents[:,i], bins=100, histtype='step', color='g')
    _ = plt.hist(samples[:,i], bins=100, histtype='step', color='b')
    plt.legend(('Reconstructed from Input', 'Sampled from Mixture Model'))
    plt.title(f'Latent {i+1}')
    plt.show()

In [ ]:
import scipy
n_L = latents.shape[-1]
fig, axs = plt.subplots(n_L, n_L, figsize=(15, 10), sharex='col', sharey='row')
for i in range(n_L):
    for j in range(n_L):
        axs[i,j].scatter(latents[:,j], latents[:,i])
        print(i, j, scipy.stats.pearsonr(latents[:,j], latents[:,i]))
fig.savefig('7L.png', dpi=600)

In [ ]:
plt.clf()

recon_energies = gas_fun(recon)

decoded_samples = experiment.model.apply({'params': experiment.state.params}, samples, rng, method=experiment.model.decode)
sampled_energies = gas_fun(decoded_samples)

org_energies = gas_fun(experiment.test_data)

threshold = 1e3
num_excluded_gmm = len(sampled_energies) - len(sampled_energies[sampled_energies<threshold])
num_excluded_nn  = len(recon_energies) - len(recon_energies[recon_energies<threshold])
print(f'Excluding Outliers GMM - {num_excluded_gmm} RECON - {num_excluded_nn}')

_ = plt.hist(org_energies[org_energies < threshold], bins=100, histtype='step', color='r')
_ = plt.hist(recon_energies[recon_energies<threshold], bins=300, histtype='step', color='g')
_ = plt.hist(sampled_energies[sampled_energies<threshold], bins=300, histtype='step', color='b')

plt.legend(('Test Data Potential', 'Reconstructed from Test', 'Sampled from Mixture Model'))
plt.xlabel('Energy (kJ/mol)')
plt.title('Comparison of Energies - GMM and NN')
plt.show()

In [ ]:
def write_traj(filename, traj_xyz): #(n conf, n_atoms*3) OR (n conf, n_atoms, 3)
    if traj_xyz.shape[-1] != 3:
        traj_xyz = traj_xyz.reshape(traj_xyz.shape[0], -1, 3)
    with md.formats.DCDTrajectoryFile(filename, 'w') as f:
        f.write(traj_xyz*10) #*10 because mdtraj loads data in nm but saves it in angstrom

my_dict = {'DA_TestData.dcd' : experiment.test_data, 'DA_ReconData.dcd' : recon, 'DA_GMM.dcd' : decoded_samples}
for key, value in zip(my_dict.keys(), my_dict.values()):
    write_traj(key, value)

In [ ]:
rmsds = atom_rmsd(experiment.test_data, recon)
plt.clf()
_ = plt.hist(rmsds, bins=100)
plt.show()
# ener_comp = scaled_pot_enr_diff(experiment.test_data, recon)
# plt.clf()
# _ = plt.hist(ener_comp[ener_comp < 5], bins=100)
# plt.show()

In [ ]:
recon_train , _ = experiment.state.apply_fn({'params':experiment.state.params}, experiment.train_data, rng)
my_dict = {'DA_TrainData.dcd' : experiment.train_data, 'DA_ReconData_train.dcd' : recon_train}
for key, value in zip(my_dict.keys(), my_dict.values()):
    write_traj(key, value)